# Freeze Strategy Comparison: Head-Only vs Progressive Unfreezing vs Full Fine-Tune

This notebook systematically compares four backbone freeze strategies for binary
oral cancer classification using MobileNetV3Large ensembles.

## Strategies
1. **Head-Only** (diagnostic baseline): Freeze entire MobileNetV3 backbone, train classifier head only
2. **Progressive Unfreeze K=3**: Head warm-up (5 ep) → unfreeze last 3 backbone blocks with discriminative LR
3. **Progressive Unfreeze K=5**: Head warm-up (5 ep) → unfreeze last 5 backbone blocks with discriminative LR
4. **Full Fine-Tune** (current production baseline): All parameters trainable from epoch 1

## Key Design Decisions
- Full 3-model ensemble per strategy (12 models total) for deployment-relevant comparison
- Warm-up epochs **don't** count against training budget (progressive gets 5 + 50 epochs)
- Bootstrap 95% CI on all key metrics for statistical rigor
- Per-source breakdown (T5/TB/TC) to diagnose domain shift
- Calibration metrics (ECE, Brier score) to assess probability quality
- Same data split, augmentation, and class weights across all strategies

## Data Sources (~4,800 images)
1. **oral_cavity_5class_dataset** (~2,838 images, mapped to binary)
2. **oral_cavity_binary_dataset** (~1,634 images)
3. **Oral-Image-small-binary-dataset** (~326 images, original only)

## 1. Configuration

In [1]:
import os
from pathlib import Path
from datetime import datetime
import json

# Dataset paths
DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
DATA_DIR_BINARY = os.path.expanduser("~/dev/oral_cavity_binary_dataset")
DATA_DIR_SMALL_BINARY = os.path.expanduser("~/dev/oral-images-dataset/Oral-Image-small-binary-dataset")

NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = os.environ.get("EXPERIMENT_NAME") or f"freeze_comparison_{timestamp}"

# Shared hyperparameters
CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "num_epochs": 50,          # Main training budget per strategy
    "patience": 12,            # Slightly more patience for fair comparison
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    "augmentation": "medical",
    "cancer_weight_multiplier": 1.5,
    "label_smoothing": 0.05,
    "dropout_rates": [0.3, 0.1, 0.2],
    "ensemble_weights": [0.30, 0.40, 0.30],

    # Strategy definitions
    "strategies": {
        "head_only": {
            "description": "Frozen backbone, train head only (diagnostic baseline)",
            "lr_head": 1e-3,       # Higher LR for head-only (fewer params, no destabilisation risk)
            "freeze": "all_backbone",
            "scheduler": "cosine",
        },
        "progressive_K3": {
            "description": "Head warm-up (5 ep) then unfreeze last 3 backbone blocks",
            "lr_head": 1e-3,
            "lr_backbone": 1e-4,   # 10x lower for backbone (discriminative LR)
            "warmup_epochs": 5,    # Does NOT count against num_epochs budget
            "unfreeze_last_k": 3,
            "scheduler": "cosine",
        },
        "progressive_K5": {
            "description": "Head warm-up (5 ep) then unfreeze last 5 backbone blocks",
            "lr_head": 1e-3,
            "lr_backbone": 1e-4,
            "warmup_epochs": 5,
            "unfreeze_last_k": 5,
            "scheduler": "cosine",
        },
        "full_finetune": {
            "description": "All parameters trainable from epoch 1 (current production baseline)",
            "lr": 1e-4,
            "scheduler": "cosine",
        },
    },
}

# Output directory
OUTPUT_DIR = NOTEBOOK_DIR / f"results/freeze_comparison/{run_name}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(exist_ok=True)
TB_LOG_DIR = OUTPUT_DIR / "tb_logs"
TB_LOG_DIR.mkdir(exist_ok=True)

# Save config
with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "run_name": run_name, "image_size": list(CONFIG["image_size"])}, f, indent=2, default=str)

print("Freeze Strategy Comparison")
print("=" * 60)
for name, strat in CONFIG["strategies"].items():
    print(f"  {name}: {strat['description']}")
print(f"\nOutput: {OUTPUT_DIR}")
print(f"Epochs per strategy: {CONFIG['num_epochs']} (+ warmup for progressive)")
print(f"Patience: {CONFIG['patience']}")
print(f"Augmentation: {CONFIG['augmentation']}")

Freeze Strategy Comparison
  head_only: Frozen backbone, train head only (diagnostic baseline)
  progressive_K3: Head warm-up (5 ep) then unfreeze last 3 backbone blocks
  progressive_K5: Head warm-up (5 ep) then unfreeze last 5 backbone blocks
  full_finetune: All parameters trainable from epoch 1 (current production baseline)

Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618
Epochs per strategy: 50 (+ warmup for progressive)
Patience: 12
Augmentation: medical


## 2. Setup and Device

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    brier_score_loss, classification_report
)
from sklearn.calibration import calibration_curve
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")   # non-interactive backend (works in headless / notebook)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print("Warning: albumentations not installed. Using basic augmentation.")
    print("Install with: pip install albumentations")

try:
    import onnx
    import onnxruntime as ort
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TENSORBOARD = True
except ImportError:
    HAS_TENSORBOARD = False

print(f"PyTorch: {torch.__version__}")
if HAS_ALBUMENTATIONS:
    print(f"Albumentations: {A.__version__}")
if HAS_ONNX:
    print(f"ONNX: {onnx.__version__}, Runtime: {ort.__version__}")
if HAS_TENSORBOARD:
    print("TensorBoard: available")
else:
    print("TensorBoard: not available (pip install tensorboard for live monitoring)")

Install with: pip install albumentations
PyTorch: 2.10.0
ONNX: 1.20.1, Runtime: 1.23.2
TensorBoard: available


In [3]:
# Device setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using Apple Silicon GPU (MPS): {device}")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {device}")
else:
    device = torch.device("cpu")
    print(f"Using CPU: {device}")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

Using Apple Silicon GPU (MPS): mps


### MPS memory (optional)

Run the cell below **during or after training** to see how much memory PyTorch is using on the MPS (GPU) backend. On Apple Silicon this is part of your Mac’s unified RAM.

In [ ]:
# MPS memory used by PyTorch (run during or after training to see current usage)
if torch.backends.mps.is_available():
    b = torch.mps.current_allocated_memory()  # bytes
    print(f"MPS current allocated: {b / 1024**2:.1f} MB")
    try:
        d = torch.mps.driver_allocated_memory()
        print(f"MPS driver allocated:  {d / 1024**2:.1f} MB")
    except Exception:
        pass
else:
    print("MPS not available")

## 3. Load Frozen Test Sets

In [4]:
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))

from frozen_test_sets import (
    build_and_save_frozen_sets_v2,
    load_frozen_sets_v2,
    get_all_train_val_for_training,
    get_t5_binary_paths_labels,
    get_tb_paths_labels,
    get_tc_paths_labels,
    get_tm_paths_labels_and_sources_v2,
)

T5, TB, TC, TM, meta = build_and_save_frozen_sets_v2(
    data_dir_5class=DATA_DIR_5CLASS,
    data_dir_binary=DATA_DIR_BINARY,
    data_dir_small_binary=DATA_DIR_SMALL_BINARY,
    out_dir=FROZEN_DIR,
)

print(f"\nFrozen test set summary:")
print(f"  T5: {len(T5)} (5-class test, binary-eligible: {meta['T5_binary_eligible']})")
print(f"  TB: {len(TB)} (binary dataset)")
print(f"  TC: {len(TC)} (small binary dataset)")
print(f"  TM: {len(TM)} (merged)")

Building frozen test sets v2...
  5-class dataset: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
  Binary dataset: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
  Small binary dataset: /Users/alokdwivedi/dev/oral-images-dataset/Oral-Image-small-binary-dataset

Frozen test sets v2 saved to /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
  T5: 534 samples (5-class test)
  TB: 246 samples (binary dataset test)
  TC: 49 samples (small binary dataset test)
  TM: 779 samples (mixed, all sources)
  binary_train_val_for_training: 1388 paths
  small_binary_train_val_for_training: 274 paths

Frozen test set summary:
  T5: 534 (5-class test, binary-eligible: 484)
  TB: 246 (binary dataset)
  TC: 49 (small binary dataset)
  TM: 779 (merged)


## 4. Load and Merge Training Data

In [5]:
def collect_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend([str(p) for p in folder.glob(ext)])
    return out

# Get binary train/val pools (excludes TB and TC)
binary_train_paths, binary_train_labels, binary_train_sources = get_all_train_val_for_training(FROZEN_DIR)

# Get 5-class train+val mapped to binary (exclude Unknown)
five_dir = Path(DATA_DIR_5CLASS)
five_cancer, five_non = [], []
for split in ["train", "val"]:
    for c in ["OCA", "OPMD"]:
        five_cancer.extend(collect_images(five_dir / split / c))
    for c in ["Benign", "Healthy"]:
        five_non.extend(collect_images(five_dir / split / c))

# Merge all sources (dedup by path)
all_paths_set = set()
all_cancer, all_non = [], []

for path, label in zip(binary_train_paths, binary_train_labels):
    if path not in all_paths_set:
        all_paths_set.add(path)
        if label == 1:
            all_cancer.append(path)
        else:
            all_non.append(path)

for p in five_cancer:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_cancer.append(p)
for p in five_non:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_non.append(p)

all_paths = all_cancer + all_non
all_labels = [1] * len(all_cancer) + [0] * len(all_non)

print(f"Merged training pool: {len(all_paths)} images")
print(f"  Cancer: {len(all_cancer)} ({len(all_cancer)/len(all_paths)*100:.1f}%)")
print(f"  Non-cancer: {len(all_non)} ({len(all_non)/len(all_paths)*100:.1f}%)")

Merged training pool: 4500 images
  Cancer: 2320 (51.6%)
  Non-cancer: 2180 (48.4%)


## 5. Train/Val Split

In [6]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)} ({sum(train_labels)} cancer, {len(train_labels)-sum(train_labels)} non-cancer)")
print(f"Val:   {len(val_paths)} ({sum(val_labels)} cancer, {len(val_labels)-sum(val_labels)} non-cancer)")

Train: 3825 (1972 cancer, 1853 non-cancer)
Val:   675 (348 cancer, 327 non-cancer)


## 6. Augmentation Pipeline

In [7]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_augmentation_pipeline(mode="medical", image_size=(224, 224)):
    """Get augmentation pipeline based on mode."""
    if HAS_ALBUMENTATIONS and mode in ["medical", "strong"]:
        if mode == "medical":
            return A.Compose([
                A.Resize(image_size[0], image_size[1]),
                A.RandomRotate90(p=0.4),
                A.Flip(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
                A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
                A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=15, val_shift_limit=12, p=0.4),
                A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),
                A.OneOf([
                    A.GaussNoise(var_limit=(10.0, 40.0), p=0.3),
                    A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.3), p=0.2),
                ], p=0.25),
                A.OneOf([
                    A.MotionBlur(blur_limit=5, p=0.3),
                    A.GaussianBlur(blur_limit=3, p=0.3),
                ], p=0.2),
                A.ImageCompression(quality_lower=70, quality_upper=95, p=0.15),
                A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
                ToTensorV2(),
            ])
    return None


def get_val_transform(image_size=(224, 224)):
    if HAS_ALBUMENTATIONS:
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None


# Fallback torchvision transforms
basic_train_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

basic_val_transform = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_aug = get_augmentation_pipeline(CONFIG["augmentation"], CONFIG["image_size"])
val_aug = get_val_transform(CONFIG["image_size"])

print(f"Augmentation: {CONFIG['augmentation']} (Albumentations: {HAS_ALBUMENTATIONS})")

Augmentation: medical (Albumentations: False)


In [8]:
class BinaryDataset(Dataset):
    """Binary classification dataset with optional Albumentations support."""

    def __init__(self, paths, labels, albu_transform=None, torch_transform=None):
        self.paths = paths
        self.labels = labels
        self.albu_transform = albu_transform
        self.torch_transform = torch_transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.albu_transform is not None:
            img = self.albu_transform(image=np.array(img))["image"]
        elif self.torch_transform is not None:
            img = self.torch_transform(img)
        return img, label


# Create datasets
if train_aug is not None:
    train_ds = BinaryDataset(train_paths, train_labels, albu_transform=train_aug)
    val_ds = BinaryDataset(val_paths, val_labels, albu_transform=val_aug)
else:
    train_ds = BinaryDataset(train_paths, train_labels, torch_transform=basic_train_transform)
    val_ds = BinaryDataset(val_paths, val_labels, torch_transform=basic_val_transform)

train_loader = DataLoader(
    train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=False,
)
val_loader = DataLoader(
    val_ds, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"],
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 120, Val batches: 22


## 7. Model Architecture

In [9]:
NUM_CLASSES = 2


class ConservativeMobileNetV3(nn.Module):
    """MobileNetV3Large with freeze/unfreeze helpers for strategy comparison."""

    def __init__(self, num_classes=2, dropout_rate=0.2):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(
                weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
            )
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)

        self.backbone.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x

    def freeze_backbone(self):
        """Freeze all backbone parameters."""
        for p in self.backbone.features.parameters():
            p.requires_grad = False

    def unfreeze_all(self):
        """Unfreeze all parameters."""
        for p in self.parameters():
            p.requires_grad = True

    def unfreeze_last_k_blocks(self, k):
        """Unfreeze last K blocks of backbone.features (Sequential of ~17 children)."""
        children = list(self.backbone.features.children())
        for layer in children[-k:]:
            for p in layer.parameters():
                p.requires_grad = True

    def get_param_groups(self, lr_head, lr_backbone=None):
        """Return parameter groups for discriminative learning rates."""
        groups = [{"params": list(self.classifier.parameters()), "lr": lr_head}]
        backbone_trainable = [p for p in self.backbone.features.parameters() if p.requires_grad]
        if backbone_trainable and lr_backbone is not None:
            groups.append({"params": backbone_trainable, "lr": lr_backbone})
        return groups

    def trainable_param_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_param_count(self):
        return sum(p.numel() for p in self.parameters())


class EnsembleModel1(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][0])


class EnsembleModel2(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][1])


class EnsembleModel3(ConservativeMobileNetV3):
    def __init__(self, num_classes=2):
        super().__init__(num_classes=num_classes, dropout_rate=CONFIG["dropout_rates"][2])
        # Slightly different architecture for diversity
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(960, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes),
        )


ENSEMBLE_CLASSES = [EnsembleModel1, EnsembleModel2, EnsembleModel3]
print(f"Ensemble: 3 models, dropout={CONFIG['dropout_rates']}")

Ensemble: 3 models, dropout=[0.3, 0.1, 0.2]


## 8. Inspect Backbone Structure

Understand what K=3 and K=5 actually unfreeze in MobileNetV3Large.

In [10]:
print("MobileNetV3Large backbone.features structure:")
print("=" * 70)

_model_inspect = ConservativeMobileNetV3()
_children = list(_model_inspect.backbone.features.children())
total_backbone_params = 0

for i, child in enumerate(_children):
    n_params = sum(p.numel() for p in child.parameters())
    total_backbone_params += n_params
    print(f"  Block {i:2d}: {child.__class__.__name__:<30s}  params={n_params:>10,}")

head_params = sum(p.numel() for p in _model_inspect.classifier.parameters())
print(f"\n  Backbone total: {total_backbone_params:,} params")
print(f"  Head total:     {head_params:,} params")
print(f"  Model total:    {total_backbone_params + head_params:,} params")

# Show what each K value unfreezes
for k in [3, 5]:
    k_params = sum(sum(p.numel() for p in c.parameters()) for c in _children[-k:])
    pct = k_params / total_backbone_params * 100
    block_names = [f"Block {len(_children)-k+j} ({_children[-k+j].__class__.__name__})"
                   for j in range(k)]
    print(f"\n  K={k}: unfreezes {k_params:,} params ({pct:.1f}% of backbone)")
    print(f"    Blocks: {', '.join(block_names)}")

del _model_inspect, _children

MobileNetV3Large backbone.features structure:
  Block  0: Conv2dNormActivation            params=       464
  Block  1: InvertedResidual                params=       464
  Block  2: InvertedResidual                params=     3,440
  Block  3: InvertedResidual                params=     4,440
  Block  4: InvertedResidual                params=    10,328
  Block  5: InvertedResidual                params=    20,992
  Block  6: InvertedResidual                params=    20,992
  Block  7: InvertedResidual                params=    32,080
  Block  8: InvertedResidual                params=    34,760
  Block  9: InvertedResidual                params=    31,992
  Block 10: InvertedResidual                params=    31,992
  Block 11: InvertedResidual                params=   214,424
  Block 12: InvertedResidual                params=   386,120
  Block 13: InvertedResidual                params=   429,224
  Block 14: InvertedResidual                params=   797,360
  Block 15: InvertedResi

## 9. Class Weights

In [11]:
classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)

cancer_multiplier = float(CONFIG.get("cancer_weight_multiplier", 1.0))
if cancer_multiplier != 1.0:
    class_weights = class_weights.clone()
    class_weights[1] = class_weights[1] * cancer_multiplier
    print(f"Applied cancer weight multiplier: {cancer_multiplier}")

print(f"Class weights: non-cancer={class_weights[0]:.4f}, cancer={class_weights[1]:.4f}")

Applied cancer weight multiplier: 1.5
Class weights: non-cancer=1.0321, cancer=1.4547


## 10. Training Functions

Four strategies implemented as separate functions, all returning the same
`(model, best_val_acc, best_val_auc, history)` interface.

In [17]:
def _make_criterion(class_weights, device, label_smoothing):
    """Create loss function (shared by all strategies)."""
    return nn.CrossEntropyLoss(
        weight=class_weights.to(device) if class_weights is not None else None,
        label_smoothing=label_smoothing,
    )


def _tb_log(writer, epoch, train_loss, train_acc, val_acc, val_auc, lr):
    """Log scalars to TensorBoard (no-op if writer is None)."""
    if writer is None:
        return
    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("acc/train", train_acc, epoch)
    writer.add_scalar("acc/val", val_acc, epoch)
    writer.add_scalar("auc/val", val_auc, epoch)
    writer.add_scalar("lr", lr, epoch)
    writer.flush()


def _train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Run one training epoch. Returns (avg_loss, accuracy%)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device).long()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = out.max(1)
        total += target.size(0)
        correct += pred.eq(target).sum().item()
    return total_loss / len(train_loader), 100.0 * correct / total


def _validate(model, val_loader, criterion, device):
    """Run validation. Returns (accuracy%, auc, avg_loss)."""
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    probs_list, targets_list = [], []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device).long()
            out = model(data)
            loss = criterion(out, target)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)
            _, pred = out.max(1)
            total += target.size(0)
            correct += pred.eq(target).sum().item()
            probs_list.extend(probs[:, 1].cpu().numpy())
            targets_list.extend(target.cpu().numpy())
    acc = 100.0 * correct / total
    auc = roc_auc_score(targets_list, probs_list)
    return acc, auc, val_loss / len(val_loader)

In [18]:
def train_head_only(model_class, model_name, train_loader, val_loader,
                    device, config, class_weights=None, tb_log_dir=None):
    """Strategy 1: Freeze entire backbone, train classifier head only.

    This is a *diagnostic baseline*. If it matches full fine-tune on AUC,
    the data isn't strong enough to justify backbone adaptation.
    """
    strat = config["strategies"]["head_only"]
    num_epochs = config["num_epochs"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = model_class(num_classes=NUM_CLASSES).to(device)
    model.freeze_backbone()

    trainable = model.trainable_param_count()
    total = model.total_param_count()
    print(f"  [{model_name}] head_only: {trainable:,} / {total:,} trainable")

    criterion = _make_criterion(class_weights, device, config.get("label_smoothing", 0.0))
    optimizer = optim.Adam(model.classifier.parameters(), lr=strat["lr_head"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_auc": [],
               "lr": [], "phase": []}

    for epoch in range(num_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr_now = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(lr_now)
        history["phase"].append("head_only")

        _tb_log(writer, epoch, avg_loss, train_acc, val_acc, val_auc, lr_now)

        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Ep {epoch+1:3d}/{num_epochs}  loss={avg_loss:.4f}  "
                  f"train={train_acc:.2f}%  val={val_acc:.2f}%  "
                  f"auc={val_auc:.4f}  lr={lr_now:.2e}")

        if patience_ctr >= config["patience"]:
            print(f"    Early stop at epoch {epoch+1}")
            break

        scheduler.step()

    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc, history

In [19]:
def train_progressive_unfreeze(model_class, model_name, train_loader, val_loader,
                              device, config, class_weights=None, unfreeze_k=3,
                              tb_log_dir=None):
    """Strategy 2/3: Head warm-up → unfreeze last K blocks with discriminative LR.

    Phase 1 (warmup_epochs):  backbone frozen, head trains at lr_head
    Phase 2 (num_epochs):     last K blocks unfrozen, fresh optimizer
                              head @ lr_head, backbone @ lr_backbone (10x lower)

    Warm-up epochs do NOT count against the main training budget, so
    progressive strategies get (warmup + num_epochs) total epochs.
    """
    strat_key = f"progressive_K{unfreeze_k}"
    strat = config["strategies"][strat_key]
    warmup_epochs = strat["warmup_epochs"]
    main_epochs = config["num_epochs"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = model_class(num_classes=NUM_CLASSES).to(device)
    model.freeze_backbone()

    criterion = _make_criterion(class_weights, device, config.get("label_smoothing", 0.0))

    # ── Phase 1: head-only warm-up ──────────────────────────────────
    trainable_p1 = model.trainable_param_count()
    print(f"  [{model_name}] {strat_key} Phase 1 (warmup {warmup_epochs} ep): "
          f"{trainable_p1:,} params")

    optimizer = optim.Adam(model.classifier.parameters(), lr=strat["lr_head"])

    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_auc": [],
               "lr": [], "phase": []}

    for epoch in range(warmup_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(strat["lr_head"])
        history["phase"].append("warmup")

        _tb_log(writer, epoch, avg_loss, train_acc, val_acc, val_auc, strat["lr_head"])

        if epoch == warmup_epochs - 1:
            print(f"    Warmup done (ep {warmup_epochs}): "
                  f"val_acc={val_acc:.2f}%, val_auc={val_auc:.4f}")

    # ── Phase 2: unfreeze last K blocks ─────────────────────────────
    model.unfreeze_last_k_blocks(unfreeze_k)
    trainable_p2 = model.trainable_param_count()
    print(f"  [{model_name}] {strat_key} Phase 2 (last {unfreeze_k} blocks, "
          f"{main_epochs} ep): {trainable_p2:,} params")

    # Fresh optimizer — clean momentum/variance for newly unfrozen params
    param_groups = model.get_param_groups(
        lr_head=strat["lr_head"],
        lr_backbone=strat["lr_backbone"],
    )
    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=main_epochs, eta_min=1e-6
    )

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0

    for epoch in range(main_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr_now = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(lr_now)
        history["phase"].append("unfreeze")

        # Global epoch = warmup + current for continuous TB x-axis
        _tb_log(writer, warmup_epochs + epoch, avg_loss, train_acc, val_acc, val_auc, lr_now)

        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            ep_display = warmup_epochs + epoch + 1
            print(f"    Ep {ep_display:3d}  loss={avg_loss:.4f}  "
                  f"train={train_acc:.2f}%  val={val_acc:.2f}%  "
                  f"auc={val_auc:.4f}  lr={lr_now:.2e}")

        if patience_ctr >= config["patience"]:
            print(f"    Early stop at epoch {warmup_epochs + epoch + 1}")
            break

        scheduler.step()

    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc, history

In [20]:
def train_full_finetune(model_class, model_name, train_loader, val_loader,
                       device, config, class_weights=None, tb_log_dir=None):
    """Strategy 4: All parameters trainable from epoch 1.

    This is the current production baseline (same as improved_binary_training.ipynb).
    """
    strat = config["strategies"]["full_finetune"]
    num_epochs = config["num_epochs"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TENSORBOARD and tb_log_dir) else None

    model = model_class(num_classes=NUM_CLASSES).to(device)
    # All params trainable by default — no freezing

    trainable = model.trainable_param_count()
    total = model.total_param_count()
    print(f"  [{model_name}] full_finetune: {trainable:,} / {total:,} trainable")

    criterion = _make_criterion(class_weights, device, config.get("label_smoothing", 0.0))
    optimizer = optim.Adam(model.parameters(), lr=strat["lr"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    history = {"train_loss": [], "train_acc": [], "val_acc": [], "val_auc": [],
               "lr": [], "phase": []}

    for epoch in range(num_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        lr_now = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["val_auc"].append(val_auc)
        history["lr"].append(lr_now)
        history["phase"].append("full")

        _tb_log(writer, epoch, avg_loss, train_acc, val_acc, val_auc, lr_now)

        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    Ep {epoch+1:3d}/{num_epochs}  loss={avg_loss:.4f}  "
                  f"train={train_acc:.2f}%  val={val_acc:.2f}%  "
                  f"auc={val_auc:.4f}  lr={lr_now:.2e}")

        if patience_ctr >= config["patience"]:
            print(f"    Early stop at epoch {epoch+1}")
            break

        scheduler.step()

    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc, history

## 11. Run All Experiments

Train 4 strategies × 3 ensemble models = **12 models** total.

**Live monitoring with TensorBoard:** Open a separate terminal and run:
```
tensorboard --logdir <output_dir>/tb_logs
```
Then open http://localhost:6006 to watch loss, AUC, accuracy, and LR curves
update in real time across all 12 runs. Use the "Runs" sidebar to filter by strategy.

### Quick checks while training
#### Healthy:
loss/train ↓
acc/train ↑
acc/val and auc/val ↑, with auc/val being the one you trust most.
Not a huge gap between acc/train and acc/val.
#### Overfitting:
loss/train keeps going down, acc/train keeps going up.
acc/val and auc/val stop improving or start going down.
Rely on “best by auc/val” and early stopping.
#### Underfitting / bad setup:
loss/train and acc/val / auc/val barely move; 
check lr and that the right parameters are trainable.
#### Summary: 
acc/val = “% correct at 0.5”; auc/val = “ranking quality, and what we use to choose the best model.” 
Use auc/val as the main number for “how well training is going” and use the other cards (loss, acc train/val, lr) to diagnose overfitting and schedule behavior.

In [ ]:
STRATEGY_NAMES = ["head_only", "progressive_K3", "progressive_K5", "full_finetune"]

all_results = {}  # strategy -> {models, histories, val_aucs, mean_val_auc}

for strategy in STRATEGY_NAMES:
    print(f"\n{'='*70}")
    print(f"STRATEGY: {strategy}")
    print(f"  {CONFIG['strategies'][strategy]['description']}")
    print(f"{'='*70}")

    models_list = []
    histories_list = []
    val_aucs = []

    for i, model_class in enumerate(ENSEMBLE_CLASSES):
        model_name = f"ensemble_{i+1}"
        print(f"\n  [{i+1}/3] {model_name}")

        # Set per-model seed for reproducibility within strategy
        torch.manual_seed(CONFIG["seed"] + i)
        np.random.seed(CONFIG["seed"] + i)

        # TensorBoard log dir: tb_logs/<strategy>/<model_name>
        run_tb_dir = TB_LOG_DIR / strategy / model_name

        if strategy == "head_only":
            model, val_acc, val_auc, hist = train_head_only(
                model_class, model_name, train_loader, val_loader,
                device, CONFIG, class_weights, tb_log_dir=run_tb_dir)
        elif strategy.startswith("progressive_K"):
            k = int(strategy.split("K")[1])
            model, val_acc, val_auc, hist = train_progressive_unfreeze(
                model_class, model_name, train_loader, val_loader,
                device, CONFIG, class_weights, unfreeze_k=k, tb_log_dir=run_tb_dir)
        elif strategy == "full_finetune":
            model, val_acc, val_auc, hist = train_full_finetune(
                model_class, model_name, train_loader, val_loader,
                device, CONFIG, class_weights, tb_log_dir=run_tb_dir)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        # Save checkpoint
        save_path = OUTPUT_DIR / "models" / f"{strategy}_{model_name}.pth"
        torch.save(model.state_dict(), save_path)

        models_list.append(model)
        histories_list.append(hist)
        val_aucs.append(val_auc)

        print(f"    Best: val_acc={val_acc:.2f}%, val_auc={val_auc:.4f}")

    all_results[strategy] = {
        "models": models_list,
        "histories": histories_list,
        "val_aucs": val_aucs,
        "mean_val_auc": float(np.mean(val_aucs)),
    }
    print(f"\n  {strategy} ensemble mean val AUC: {np.mean(val_aucs):.4f}")

# Reset seed
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

print("\n" + "=" * 70)
print("ALL STRATEGIES TRAINED")
print("=" * 70)
for s in STRATEGY_NAMES:
    r = all_results[s]
    aucs_str = ", ".join(f"{a:.4f}" for a in r["val_aucs"])
    print(f"  {s:<22s}: mean_val_auc={r['mean_val_auc']:.4f}  [{aucs_str}]")

if HAS_TENSORBOARD:
    print(f"\nTensorBoard logs: {TB_LOG_DIR}")
    print(f"  Run in a separate terminal:")
    print(f"  tensorboard --logdir '{TB_LOG_DIR}'")


STRATEGY: head_only
  Frozen backbone, train head only (diagnostic baseline)

  [1/3] ensemble_1
  [ensemble_1] head_only: 246,530 / 3,218,482 trainable
    Ep   1/50  loss=0.6099  train=64.92%  val=68.89%  auc=0.7851  lr=1.00e-03
    Ep   5/50  loss=0.5063  train=74.35%  val=70.96%  auc=0.8356  lr=9.84e-04


In [ ]:
# Launch TensorBoard (run this cell before or during training to monitor live)
# Command to run in a separate terminal if you prefer:
#   tensorboard --logdir '<path_below>'

import subprocess
import sys

logdir = str(TB_LOG_DIR)
print(f"Log directory: {logdir}")
print(f"\nTo run in a separate terminal:\n  tensorboard --logdir '{logdir}'")

# Start TensorBoard in the background (so this cell returns and notebook doesn't block)
try:
    proc = subprocess.Popen(
        [sys.executable, "-m", "tensorboard", "--logdir", logdir, "--port", "6006"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print(f"\nTensorBoard started in background (PID {proc.pid}). Open http://localhost:6006")
except Exception as e:
    print(f"\nCould not start TensorBoard: {e}")
    print("Run the command above in a terminal instead.")

## 12. Evaluation on Frozen Test Sets

Evaluate each strategy's 3-model ensemble on T5, TB, TC, TM using
weighted average of logits (same as production inference).

In [30]:
def preprocess_for_eval(path, image_size=(224, 224)):
    """Preprocess image for PyTorch model inference."""
    img = Image.open(path).convert("RGB").resize(image_size)
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - IMAGENET_MEAN) / IMAGENET_STD
    return torch.tensor(np.transpose(arr, (2, 0, 1))).unsqueeze(0).float()


def evaluate_ensemble_pytorch(models, paths, labels, device,
                              ensemble_weights=None):
    """Evaluate a PyTorch ensemble on a test set."""
    if ensemble_weights is None:
        ensemble_weights = CONFIG["ensemble_weights"]

    for m in models:
        m.eval()
        m.to(device)

    y_true, y_prob = [], []

    with torch.no_grad():
        for path, label in zip(paths, labels):
            x = preprocess_for_eval(path).to(device)
            logits = sum(
                m(x).cpu().numpy() * w
                for m, w in zip(models, ensemble_weights)
            )
            exp_logits = np.exp(logits - np.max(logits))
            prob_cancer = float(exp_logits[0, 1] / exp_logits.sum())
            y_true.append(label)
            y_prob.append(prob_cancer)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "n": len(y_true),
        "accuracy": (tp + tn) / len(y_true),
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "confusion": {"tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)},
        "y_true": y_true,
        "y_prob": y_prob,
    }


def sensitivity_at_specificity(y_true, y_prob, target_spec):
    """Find sensitivity at a target specificity level."""
    thresholds = np.linspace(0, 1, 1001)
    best_sens = 0.0
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
        spec = tn / max(tn + fp, 1)
        sens = tp / max(tp + fn, 1)
        if spec >= target_spec:
            best_sens = max(best_sens, sens)
    return best_sens


def compute_ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bin_boundaries[i]) & (y_prob < bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_conf = y_prob[mask].mean()
        bin_acc = y_true[mask].mean()
        ece += mask.sum() / len(y_true) * abs(bin_conf - bin_acc)
    return ece

In [31]:
# Load frozen test set paths
t5_paths, t5_labels = get_t5_binary_paths_labels(FROZEN_DIR)
tb_paths, tb_labels = get_tb_paths_labels(FROZEN_DIR)
tc_paths, tc_labels = get_tc_paths_labels(FROZEN_DIR)
tm_paths, tm_labels, tm_sources = get_tm_paths_labels_and_sources_v2(FROZEN_DIR)

test_sets = {
    "T5": (t5_paths, t5_labels),
    "TB": (tb_paths, tb_labels),
    "TC": (tc_paths, tc_labels),
    "TM": (tm_paths, tm_labels),
}

eval_results = {}  # strategy -> test_set_name -> metrics

print("Evaluating all strategies on frozen test sets...")
print("=" * 90)

for strategy in STRATEGY_NAMES:
    print(f"\n  {strategy}:")
    models = all_results[strategy]["models"]
    eval_results[strategy] = {}

    for set_name, (paths, labels) in test_sets.items():
        if len(paths) == 0:
            continue
        metrics = evaluate_ensemble_pytorch(models, paths, labels, device)
        eval_results[strategy][set_name] = metrics

        # Also compute clinical metrics
        metrics["sens_at_90spec"] = sensitivity_at_specificity(
            metrics["y_true"], metrics["y_prob"], 0.90)
        metrics["sens_at_85spec"] = sensitivity_at_specificity(
            metrics["y_true"], metrics["y_prob"], 0.85)
        metrics["ece"] = compute_ece(metrics["y_true"], metrics["y_prob"])

        print(f"    {set_name} (n={metrics['n']:>4d}): "
              f"Acc={metrics['accuracy']*100:.2f}%  "
              f"Sens={metrics['sensitivity']*100:.2f}%  "
              f"Spec={metrics['specificity']*100:.2f}%  "
              f"AUROC={metrics['auroc']:.4f}  "
              f"Brier={metrics['brier']:.4f}")

Evaluating all strategies on frozen test sets...

  head_only:
    T5 (n= 484): Acc=73.97%  Sens=82.86%  Spec=64.85%  AUROC=0.8259  Brier=0.1769
    TB (n= 246): Acc=92.28%  Sens=92.48%  Spec=92.04%  AUROC=0.9718  Brier=0.0603
    TC (n=  49): Acc=85.71%  Sens=100.00%  Spec=72.00%  AUROC=0.9900  Brier=0.1017
    TM (n= 779): Acc=80.49%  Sens=87.06%  Spec=73.47%  AUROC=0.8952  Brier=0.1353

  progressive_K3:
    T5 (n= 484): Acc=75.00%  Sens=84.08%  Spec=65.69%  AUROC=0.8559  Brier=0.1668
    TB (n= 246): Acc=92.28%  Sens=90.98%  Spec=93.81%  AUROC=0.9843  Brier=0.0495
    TC (n=  49): Acc=79.59%  Sens=100.00%  Spec=60.00%  AUROC=1.0000  Brier=0.1128
    TM (n= 779): Acc=80.74%  Sens=87.31%  Spec=73.74%  AUROC=0.9149  Brier=0.1263

  progressive_K5:
    T5 (n= 484): Acc=79.34%  Sens=80.41%  Spec=78.24%  AUROC=0.8830  Brier=0.1456
    TB (n= 246): Acc=94.72%  Sens=92.48%  Spec=97.35%  AUROC=0.9836  Brier=0.0461
    TC (n=  49): Acc=85.71%  Sens=100.00%  Spec=72.00%  AUROC=0.9983  Brier=0

## 13. Per-Source Breakdown on TM

Split TM metrics by data source (A=5-class, B=binary, C=small-binary) to
diagnose domain shift. A good strategy should have **small gaps** between sources.

In [32]:
print("Per-Source Breakdown on TM")
print("=" * 90)

source_labels_map = {"A": "5-class", "B": "Binary", "C": "SmallBin"}
tm_sources_arr = np.array(tm_sources)

per_source_results = {}  # strategy -> source -> {auroc, sens, spec}

for strategy in STRATEGY_NAMES:
    print(f"\n  {strategy}:")
    per_source_results[strategy] = {}

    tm_metrics = eval_results[strategy]["TM"]
    y_true = tm_metrics["y_true"]
    y_prob = tm_metrics["y_prob"]

    for source_code in ["A", "B", "C"]:
        mask = tm_sources_arr == source_code
        if mask.sum() == 0:
            continue

        src_true = y_true[mask]
        src_prob = y_prob[mask]
        src_pred = (src_prob >= 0.5).astype(int)

        if len(np.unique(src_true)) < 2:
            print(f"    Source {source_code} ({source_labels_map[source_code]}): "
                  f"n={mask.sum()}, only one class — skipping AUC")
            continue

        tn, fp, fn, tp = confusion_matrix(src_true, src_pred).ravel()
        auroc = roc_auc_score(src_true, src_prob)
        sens = tp / max(tp + fn, 1)
        spec = tn / max(tn + fp, 1)

        per_source_results[strategy][source_code] = {
            "n": int(mask.sum()), "auroc": auroc,
            "sensitivity": sens, "specificity": spec,
        }

        print(f"    Source {source_code} ({source_labels_map[source_code]:>8s}): "
              f"n={mask.sum():>4d}  "
              f"Sens={sens*100:.1f}%  Spec={spec*100:.1f}%  AUROC={auroc:.4f}")

# Source gap diagnostic
print(f"\n  Source stability (|AUROC_A - AUROC_B| gap, lower is better):")
for strategy in STRATEGY_NAMES:
    ps = per_source_results[strategy]
    if "A" in ps and "B" in ps:
        gap = abs(ps["A"]["auroc"] - ps["B"]["auroc"])
        print(f"    {strategy:<22s}: A={ps['A']['auroc']:.4f}  B={ps['B']['auroc']:.4f}  "
              f"gap={gap:.4f}")

Per-Source Breakdown on TM

  head_only:
    Source A ( 5-class): n= 484  Sens=82.9%  Spec=64.9%  AUROC=0.8259
    Source B (  Binary): n= 246  Sens=92.5%  Spec=92.0%  AUROC=0.9718
    Source C (SmallBin): n=  49  Sens=100.0%  Spec=72.0%  AUROC=0.9900

  progressive_K3:
    Source A ( 5-class): n= 484  Sens=84.1%  Spec=65.7%  AUROC=0.8559
    Source B (  Binary): n= 246  Sens=91.0%  Spec=93.8%  AUROC=0.9843
    Source C (SmallBin): n=  49  Sens=100.0%  Spec=60.0%  AUROC=1.0000

  progressive_K5:
    Source A ( 5-class): n= 484  Sens=80.4%  Spec=78.2%  AUROC=0.8830
    Source B (  Binary): n= 246  Sens=92.5%  Spec=97.3%  AUROC=0.9836
    Source C (SmallBin): n=  49  Sens=100.0%  Spec=72.0%  AUROC=0.9983

  full_finetune:
    Source A ( 5-class): n= 484  Sens=82.0%  Spec=79.9%  AUROC=0.8810
    Source B (  Binary): n= 246  Sens=89.5%  Spec=98.2%  AUROC=0.9774
    Source C (SmallBin): n=  49  Sens=100.0%  Spec=88.0%  AUROC=1.0000

  Source stability (|AUROC_A - AUROC_B| gap, lower is bett

## 14. Bootstrap Confidence Intervals

95% CI (1000 resamples) for AUROC and sensitivity on T5, TB, and TM.
TC is excluded — with only ~49 samples, bootstrap CIs are unreliable.

In [33]:
def bootstrap_metric(y_true, y_prob, metric_fn, n_bootstrap=1000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for a metric."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []

    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        bt_true = y_true[idx]
        bt_prob = y_prob[idx]
        if len(np.unique(bt_true)) < 2:
            continue
        try:
            scores.append(metric_fn(bt_true, bt_prob))
        except Exception:
            continue

    if len(scores) < 100:
        return np.nan, np.nan, np.nan

    scores = np.array(scores)
    alpha = (1 - ci) / 2
    return (np.mean(scores),
            np.percentile(scores, alpha * 100),
            np.percentile(scores, (1 - alpha) * 100))


def _sens_at_050(yt, yp):
    ypr = (yp >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, ypr).ravel()
    return tp / max(tp + fn, 1)


print("Bootstrap 95% CI (1000 resamples)")
print("=" * 100)
print(f"{'Strategy':<22s} {'Set':<5s} "
      f"{'AUROC':>8s} {'95% CI':>20s}    "
      f"{'Sens':>6s} {'95% CI':>20s}")
print("-" * 100)

bootstrap_results = {}

for strategy in STRATEGY_NAMES:
    bootstrap_results[strategy] = {}

    for set_name in ["T5", "TB", "TM"]:
        if set_name not in eval_results[strategy]:
            continue
        m = eval_results[strategy][set_name]

        auc_mean, auc_lo, auc_hi = bootstrap_metric(
            m["y_true"], m["y_prob"], roc_auc_score)
        sens_mean, sens_lo, sens_hi = bootstrap_metric(
            m["y_true"], m["y_prob"], _sens_at_050)

        bootstrap_results[strategy][set_name] = {
            "auroc": (float(auc_mean), float(auc_lo), float(auc_hi)),
            "sensitivity": (float(sens_mean), float(sens_lo), float(sens_hi)),
        }

        print(f"{strategy:<22s} {set_name:<5s} "
              f"{auc_mean:>8.4f} [{auc_lo:.4f}, {auc_hi:.4f}]    "
              f"{sens_mean:>6.4f} [{sens_lo:.4f}, {sens_hi:.4f}]")

print(f"\n  Note: TC (n={len(tc_labels)}) omitted — too small for reliable bootstrap CI.")

Bootstrap 95% CI (1000 resamples)
Strategy               Set      AUROC               95% CI      Sens               95% CI
----------------------------------------------------------------------------------------------------
head_only              T5      0.8266 [0.7874, 0.8609]    0.8292 [0.7796, 0.8775]
head_only              TB      0.9718 [0.9486, 0.9886]    0.9247 [0.8788, 0.9641]
head_only              TM      0.8955 [0.8739, 0.9155]    0.8709 [0.8370, 0.9013]
progressive_K3         T5      0.8565 [0.8224, 0.8883]    0.8408 [0.7934, 0.8872]
progressive_K3         TB      0.9844 [0.9709, 0.9941]    0.9102 [0.8603, 0.9549]
progressive_K3         TM      0.9151 [0.8973, 0.9332]    0.8728 [0.8381, 0.9039]
progressive_K5         T5      0.8835 [0.8531, 0.9122]    0.8041 [0.7511, 0.8549]
progressive_K5         TB      0.9836 [0.9670, 0.9950]    0.9250 [0.8768, 0.9669]
progressive_K5         TM      0.9291 [0.9118, 0.9455]    0.8553 [0.8218, 0.8868]
full_finetune          T5      0.8814

## 15. Calibration Analysis

Brier score, ECE, and reliability diagrams on TM for each strategy.
Better calibration → more trustworthy probability outputs in the app.

In [35]:
fig, axes = plt.subplots(1, len(STRATEGY_NAMES), figsize=(5 * len(STRATEGY_NAMES), 5), sharey=True)

print("Calibration Metrics (on TM)")
print("=" * 60)
print(f"{'Strategy':<22s} {'Brier':>8s} {'ECE':>8s}")
print("-" * 40)

for ax, strategy in zip(axes, STRATEGY_NAMES):
    m = eval_results[strategy]["TM"]
    y_true = m["y_true"]
    y_prob = m["y_prob"]

    brier = brier_score_loss(y_true, y_prob)
    ece = compute_ece(y_true, y_prob)
    print(f"{strategy:<22s} {brier:>8.4f} {ece:>8.4f}")

    # Reliability diagram
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect")
    ax.plot(prob_pred, prob_true, "s-", color="tab:blue",
            label=f"Brier={brier:.4f}\nECE={ece:.4f}")
    ax.set_xlabel("Mean predicted probability")
    ax.set_title(strategy, fontsize=10)
    ax.legend(loc="upper left", fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect("equal")

axes[0].set_ylabel("Fraction of positives")
fig.suptitle("Reliability Diagrams (TM)", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "calibration_reliability.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved: {OUTPUT_DIR / 'plots' / 'calibration_reliability.png'}")

Calibration Metrics (on TM)
Strategy                  Brier      ECE
----------------------------------------
head_only                0.1353   0.0561
progressive_K3           0.1263   0.0599
progressive_K5           0.1099   0.0476
full_finetune            0.1094   0.0578

Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/plots/calibration_reliability.png


## 16. Training Dynamics

Compare learning curves across strategies to visualise convergence speed,
overfitting gap (train-val accuracy delta), and LR schedules.

In [36]:
colors = {
    "head_only":        "tab:green",
    "progressive_K3":   "tab:blue",
    "progressive_K5":   "tab:purple",
    "full_finetune":    "tab:orange",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Plot 1: Val AUC (mean of 3 ensemble models) ──
ax = axes[0, 0]
for strategy in STRATEGY_NAMES:
    histories = all_results[strategy]["histories"]
    max_len = max(len(h["val_auc"]) for h in histories)
    mean_auc = []
    for ep in range(max_len):
        vals = [h["val_auc"][ep] for h in histories if ep < len(h["val_auc"])]
        mean_auc.append(np.mean(vals))
    ax.plot(range(1, len(mean_auc) + 1), mean_auc,
            label=strategy, color=colors[strategy], linewidth=2)
ax.set_xlabel("Epoch (including warmup)")
ax.set_ylabel("Val AUC (mean of 3)")
ax.set_title("Validation AUC")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 2: Train Loss ──
ax = axes[0, 1]
for strategy in STRATEGY_NAMES:
    histories = all_results[strategy]["histories"]
    max_len = max(len(h["train_loss"]) for h in histories)
    mean_loss = []
    for ep in range(max_len):
        vals = [h["train_loss"][ep] for h in histories if ep < len(h["train_loss"])]
        mean_loss.append(np.mean(vals))
    ax.plot(range(1, len(mean_loss) + 1), mean_loss,
            label=strategy, color=colors[strategy], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Loss (mean of 3)")
ax.set_title("Training Loss")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 3: Overfitting Gap (train_acc - val_acc) ──
ax = axes[1, 0]
for strategy in STRATEGY_NAMES:
    histories = all_results[strategy]["histories"]
    max_len = max(len(h["train_acc"]) for h in histories)
    mean_gap = []
    for ep in range(max_len):
        t_accs = [h["train_acc"][ep] for h in histories
                  if ep < len(h["train_acc"]) and h["train_acc"][ep] > 0]
        v_accs = [h["val_acc"][ep] for h in histories if ep < len(h["val_acc"])]
        if t_accs and v_accs:
            mean_gap.append(np.mean(t_accs) - np.mean(v_accs))
    if mean_gap:
        ax.plot(range(1, len(mean_gap) + 1), mean_gap,
                label=strategy, color=colors[strategy], linewidth=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Acc - Val Acc (%)")
ax.set_title("Overfitting Gap (lower is better)")
ax.axhline(y=0, color="k", linestyle="--", alpha=0.3)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Plot 4: Learning Rate ──
ax = axes[1, 1]
for strategy in STRATEGY_NAMES:
    h = all_results[strategy]["histories"][0]  # Model 1 only
    ax.plot(range(1, len(h["lr"]) + 1), h["lr"],
            label=strategy, color=colors[strategy], linewidth=2)
    # Annotate unfreeze point for progressive strategies
    if "warmup" in h["phase"]:
        wu_end = h["phase"].count("warmup")
        ax.axvline(x=wu_end, color=colors[strategy], linestyle=":", alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("LR Schedule (model 1)")
ax.set_yscale("log")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle("Training Dynamics Comparison", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "training_dynamics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUTPUT_DIR / 'plots' / 'training_dynamics.png'}")

Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/plots/training_dynamics.png


## 17. Comparison Summary

Side-by-side table of all strategies with automated recommendation.

**Selection criterion:** highest `min(T5_AUC, TB_AUC)` with TM sensitivity >= 80%.
This rewards **cross-source stability** — a strategy that does well on all sources
rather than excelling on one and failing on another.

In [37]:
print("=" * 120)
print("COMPARISON SUMMARY")
print("=" * 120)

header = (f"{'Strategy':<22s} {'T5 AUC':>8s} {'TB AUC':>8s} {'TC AUC':>8s} "
          f"{'TM AUC':>8s} {'TM Sens':>8s} {'TM Spec':>8s} "
          f"{'S@90Sp':>7s} {'Brier':>7s} {'ECE':>7s} {'Params':>10s}")
print(header)
print("-" * 120)

summary_rows = []

for strategy in STRATEGY_NAMES:
    m0 = all_results[strategy]["models"][0]
    n_params = m0.trainable_param_count()

    row = {"strategy": strategy, "trainable_params": n_params}
    for set_name in ["T5", "TB", "TC", "TM"]:
        if set_name in eval_results[strategy]:
            r = eval_results[strategy][set_name]
            row[f"{set_name}_auroc"] = r["auroc"]
            row[f"{set_name}_sens"] = r["sensitivity"]
            row[f"{set_name}_spec"] = r["specificity"]

    tm = eval_results[strategy].get("TM", {})
    row["brier"] = tm.get("brier", float("nan"))
    row["ece"] = tm.get("ece", float("nan"))
    row["sens_at_90spec"] = tm.get("sens_at_90spec", float("nan"))

    summary_rows.append(row)

    print(f"{strategy:<22s} "
          f"{row.get('T5_auroc', float('nan')):>8.4f} "
          f"{row.get('TB_auroc', float('nan')):>8.4f} "
          f"{row.get('TC_auroc', float('nan')):>8.4f} "
          f"{row.get('TM_auroc', float('nan')):>8.4f} "
          f"{row.get('TM_sens', float('nan'))*100:>7.2f}% "
          f"{row.get('TM_spec', float('nan'))*100:>7.2f}% "
          f"{row.get('sens_at_90spec', float('nan'))*100:>6.1f}% "
          f"{row.get('brier', float('nan')):>7.4f} "
          f"{row.get('ece', float('nan')):>7.4f} "
          f"{n_params:>10,}")

# ── Automated Recommendation ──────────────────────────────────────
print("\n" + "=" * 120)
print("RECOMMENDATION")
print("=" * 120)

best_strategy = None
best_score = -1

for row in summary_rows:
    t5a = row.get("T5_auroc", 0)
    tba = row.get("TB_auroc", 0)
    tms = row.get("TM_sens", 0)
    min_auc = min(t5a, tba)
    if tms >= 0.80 and min_auc > best_score:
        best_score = min_auc
        best_strategy = row["strategy"]

if best_strategy:
    print(f"\n  Winner: {best_strategy}")
    print(f"  Criterion: highest min(T5_AUC, TB_AUC) with TM sensitivity >= 80%")
    print(f"  Score: min(T5, TB) AUC = {best_score:.4f}")
else:
    print("\n  No strategy met the minimum sensitivity threshold (80%).")
    print("  Review results manually.")

# ── Diagnostic: Head-only vs Full interpretation ──────────────────
head_auc = eval_results["head_only"]["TM"]["auroc"]
full_auc = eval_results["full_finetune"]["TM"]["auroc"]
gap = full_auc - head_auc

print(f"\n  Diagnostic: head_only vs full_finetune TM AUROC gap = {gap:+.4f}")
if abs(gap) < 0.02:
    print("  --> Gap < 0.02: backbone adaptation provides minimal benefit.")
    print("      The dataset may not be strong enough, or domain shift from")
    print("      ImageNet is not being effectively bridged. Consider: more data,")
    print("      better augmentation, or self-supervised pre-training.")
elif gap > 0.02:
    print("  --> Full fine-tune outperforms head-only: backbone adaptation helps.")
    print("      The question is HOW MUCH to adapt (progressive vs full).")
else:
    print("  --> Head-only outperforms full fine-tune: strong overfitting signal.")
    print("      Progressive unfreezing or head-only is likely the right approach.")

# ── Source stability ──────────────────────────────────────────────
print(f"\n  Source stability (T5 vs TB AUROC gap, lower = more stable):")
for strategy in STRATEGY_NAMES:
    t5a = eval_results[strategy]["T5"]["auroc"]
    tba = eval_results[strategy]["TB"]["auroc"]
    marker = " <-- best" if strategy == best_strategy else ""
    print(f"    {strategy:<22s}: T5={t5a:.4f}  TB={tba:.4f}  "
          f"gap={abs(t5a - tba):.4f}{marker}")

# ── Bootstrap overlap check ──────────────────────────────────────
print(f"\n  Bootstrap CI overlap check (TM AUROC):")
for strategy in STRATEGY_NAMES:
    if strategy in bootstrap_results and "TM" in bootstrap_results[strategy]:
        ci = bootstrap_results[strategy]["TM"]["auroc"]
        print(f"    {strategy:<22s}: {ci[0]:.4f} [{ci[1]:.4f}, {ci[2]:.4f}]")

# Save results
summary_json = []
for row in summary_rows:
    clean = {k: (float(v) if isinstance(v, (np.floating, float)) else v)
             for k, v in row.items()}
    summary_json.append(clean)

save_data = {
    "strategies": summary_json,
    "recommendation": best_strategy,
    "bootstrap": {
        s: {ts: {m: [float(x) for x in vals]
                  for m, vals in metrics.items()}
            for ts, metrics in ts_data.items()}
        for s, ts_data in bootstrap_results.items()
    },
}
with open(OUTPUT_DIR / "comparison_summary.json", "w") as f:
    json.dump(save_data, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR / 'comparison_summary.json'}")

COMPARISON SUMMARY
Strategy                 T5 AUC   TB AUC   TC AUC   TM AUC  TM Sens  TM Spec  S@90Sp   Brier     ECE     Params
------------------------------------------------------------------------------------------------------------------------
head_only                0.8259   0.9718   0.9900   0.8952   87.06%   73.47%   70.1%  0.1353  0.0561    246,530
progressive_K3           0.8559   0.9843   1.0000   0.9149   87.31%   73.74%   73.6%  0.1263  0.0599  1,996,770
progressive_K5           0.8830   0.9836   0.9983   0.9291   85.57%   83.55%   78.9%  0.1099  0.0476  2,812,114
full_finetune            0.8810   0.9774   1.0000   0.9235   85.57%   85.94%   82.6%  0.1094  0.0578  3,218,482

RECOMMENDATION

  Winner: progressive_K5
  Criterion: highest min(T5_AUC, TB_AUC) with TM sensitivity >= 80%
  Score: min(T5, TB) AUC = 0.8830

  Diagnostic: head_only vs full_finetune TM AUROC gap = +0.0282
  --> Full fine-tune outperforms head-only: backbone adaptation helps.
      The question i

## 18. ONNX Export (Winning Strategy)

Export the winning strategy's 3-model ensemble to ONNX for deployment.

In [38]:
def convert_to_onnx(model, onnx_path, device, input_shape=(1, 3, 224, 224)):
    """Convert PyTorch model to single-file ONNX."""
    model.eval()
    model.to(device)
    dummy = torch.randn(input_shape).to(device)

    torch.onnx.export(
        model, dummy, onnx_path,
        export_params=True,
        opset_version=18,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )

    if HAS_ONNX:
        onnx_model = onnx.load(str(onnx_path))
        onnx.save_model(onnx_model, str(onnx_path), save_as_external_data=False)
        data_file = Path(str(onnx_path) + ".data")
        if data_file.exists():
            data_file.unlink()


if best_strategy:
    print(f"Exporting ONNX for winning strategy: {best_strategy}")
    onnx_out = OUTPUT_DIR / "onnx"
    onnx_out.mkdir(exist_ok=True)

    for i, model in enumerate(all_results[best_strategy]["models"]):
        name = f"ensemble_model_{i+1}"
        onnx_path = onnx_out / f"{name}.onnx"
        convert_to_onnx(model, onnx_path, device)
        size_mb = onnx_path.stat().st_size / 1024 / 1024
        print(f"  {name}.onnx ({size_mb:.2f} MB)")

    print(f"\nONNX models saved to: {onnx_out}")
    print(f"\nTo deploy:")
    for i in range(3):
        print(f"  cp {onnx_out}/ensemble_model_{i+1}.onnx <repo>/assets/binary_ensemble/")
else:
    print("No winning strategy identified. Review results manually before exporting.")

Exporting ONNX for winning strategy: progressive_K5


W0209 15:03:38.805000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:38.805000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:38.806000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0209 15:03:38.806000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel1([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.


W0209 15:03:41.829000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:41.830000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:41.830000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0209 15:03:41.831000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


  ensemble_model_1.onnx (12.61 MB)
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.


W0209 15:03:44.714000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:44.715000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0209 15:03:44.715000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0209 15:03:44.716000 54216 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


  ensemble_model_2.onnx (12.61 MB)
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EnsembleModel3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
  ensemble_model_3.onnx (13.55 MB)

ONNX models saved to: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/onnx

To deploy:
  cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/onnx/ensemble_model_1.onnx <repo>/assets/binary_ensemble/
  cp /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/

## 19. Final Summary

In [39]:
print("=" * 70)
print("FREEZE STRATEGY COMPARISON COMPLETE")
print("=" * 70)

print(f"\nRun: {run_name}")
print(f"Output: {OUTPUT_DIR}")
print(f"Strategies tested: {len(STRATEGY_NAMES)}")
print(f"Models trained: {len(STRATEGY_NAMES) * 3}")

if best_strategy:
    bm = eval_results[best_strategy]
    print(f"\nRecommended strategy: {best_strategy}")
    print(f"  TM: AUROC={bm['TM']['auroc']:.4f}  "
          f"Sens={bm['TM']['sensitivity']*100:.2f}%  "
          f"Spec={bm['TM']['specificity']*100:.2f}%")
    if "T5" in bm:
        print(f"  T5: AUROC={bm['T5']['auroc']:.4f}")
    if "TB" in bm:
        print(f"  TB: AUROC={bm['TB']['auroc']:.4f}")
else:
    print("\n  No clear winner — review comparison_summary.json manually.")

print(f"\nFiles:")
print(f"  Config:  {OUTPUT_DIR / 'config.json'}")
print(f"  Summary: {OUTPUT_DIR / 'comparison_summary.json'}")
print(f"  Models:  {OUTPUT_DIR / 'models'}")
print(f"  Plots:   {OUTPUT_DIR / 'plots'}")
if best_strategy:
    print(f"  ONNX:    {OUTPUT_DIR / 'onnx'}")

print(f"\nNext steps:")
print(f"  1. If winner is progressive, consider testing K=2 or K=6 to bracket further.")
print(f"  2. Try augmentation intensity sweep with the winning freeze strategy.")
print(f"  3. Consider self-supervised pre-training to build a better backbone.")

FREEZE STRATEGY COMPARISON COMPLETE

Run: freeze_comparison_20260208_121618
Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618
Strategies tested: 4
Models trained: 12

Recommended strategy: progressive_K5
  TM: AUROC=0.9291  Sens=85.57%  Spec=83.55%
  T5: AUROC=0.8830
  TB: AUROC=0.9836

Files:
  Config:  /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/config.json
  Summary: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/comparison_summary.json
  Models:  /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/freeze_comparison/freeze_comparison_20260208_121618/models
  Plots:   /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/resul